# 🚀 SemanticCodec v1.0.0 — Google Colab A100 爆速起動ノートブック

Google Colab Pro / AI Pro の **NVIDIA A100 (40GB / 80GB VRAM)** 上で `SemanticCodec v1.0.0` を **High-VRAM 全モデル GPU 常駐モード（TF32 + SDPA FlashAttention 有効）** で動作させる専用ノートブックです。

---
### ⚠️ 重要：課金（コンピューティングユニット消費）の開始・停止タイミングについて
- **いつ課金カウントが始まる？**
  - このノートブックを開いただけ・閲覧しているだけでは **一切消費されません**。
  - 右上の **「接続 (Connect)」** を押した瞬間、または **最初のコードセルの「▶（実行）」を押して A100 ランタイムが割り当てられた瞬間** から時間比例でコンピューティングユニットの消費が始まります。
  - ※ A100 に接続中は、処理をしていない待機中もユニットが消費され続けます。
- **どうすれば課金カウントが止まる？**
  - ブラウザのタブを閉じただけでは裏でランタイムが動き続けてしまいます。
  - テストが終わったら、必ず **一番下の「🛑 ステップ 5：ランタイムを完全停止して課金カウントを終了する」セルを実行** するか、上部メニューの **`ランタイム` → `ランタイムを接続解除して削除`** をクリックしてください。それで即座に消費がゼロになります。

## ステップ 1：GPU (A100) の確認とリポジトリ・パッケージの準備
上部メニューの **`ランタイム` → `ランタイムのタイプを変更`** で **ハードウェアアクセラレータ：`A100 GPU`** を選択してから、下の「▶」を押してください。

In [ ]:
import os, subprocess, torch

print('=== 1. GPU 環境チェック ===')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f'✅ GPU 検出成功: {gpu_name} ({vram_gb} GB VRAM)')
    os.environ['SEMIC_HIGH_VRAM'] = '1'
else:
    print('⚠️ GPU が有効になっていません！「ランタイムのタイプを変更」から A100 GPU を選択してください。')

print('\n=== 2. SemanticCodec リポジトリの取得 ===')
REPO_DIR = '/content/SemanticCodec_repo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Lil4AI/SemanticCodec.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
WORK_DIR = os.path.join(REPO_DIR, 'SemanticCodec') if os.path.exists(os.path.join(REPO_DIR, 'SemanticCodec', 'app.py')) else REPO_DIR
%cd {WORK_DIR}
print(f'📂 作業ディレクトリ: {os.getcwd()}')

print('\n=== 3. 追加パッケージのインストール ===')
# NumPy 2.x の破壊的変更による transformers/rembg の競合を防ぐため numpy<2 を指定
!pip install -q "numpy<2" diffusers transformers accelerate safetensors peft rembg onnxruntime-gpu piexif flask werkzeug
print('✅ セットアップ完了！')

## （オプション）ステップ 1.5：GitHub プッシュ前のローカル最新ファイルを直接アップロードする場合
もしまだ GitHub（`Lil4AI/SemanticCodec`）に v1.0.0 のファイルをプッシュしていない場合は、下のセルを実行して PC の `Git_update` フォルダ内のファイル（`codec_engine.py`, `app.py`, `cli.py`, `index.html`, `app.js`, `style.css`）を選択してアップロードできます。すでに GitHub にプッシュ済みの場合はスキップしてステップ 2 へ進んでください。

In [ ]:
import os, shutil
from google.colab import files

cwd = os.getcwd()
os.makedirs(os.path.join(cwd, 'templates'), exist_ok=True)
os.makedirs(os.path.join(cwd, 'static'), exist_ok=True)

print('Git_update フォルダ内のファイル（複数選択可）を選択してください（スキップする場合はキャンセル）:')
uploaded = files.upload()
for fname in uploaded.keys():
    if fname.endswith('.html'):
        shutil.move(fname, os.path.join(cwd, 'templates/index.html'))
        print('  -> templates/index.html に配置しました')
    elif fname.endswith('.js'):
        shutil.move(fname, os.path.join(cwd, 'static/app.js'))
        print('  -> static/app.js に配置しました')
    elif fname.endswith('.css'):
        shutil.move(fname, os.path.join(cwd, 'static/style.css'))
        print('  -> static/style.css に配置しました')
    else:
        shutil.move(fname, os.path.join(cwd, fname))
        print(f'  -> {fname} を配置しました')

## ステップ 2：A100 爆速モードで全 AI モデルを VRAM に事前ロード（ウォームアップ）
A100 の大容量 VRAM（40GB / 80GB）を活かして、エンコーダ（U2-Net / BLIP / Depth-Anything-V2 / SegFormer）とデコーダ（SD-Inpaint + Multi-ControlNet + LCM-LoRA）をすべて GPU メモリに常駐させます。

In [ ]:
import codec_engine as engine

models = engine.ModelManager()
print('Status before load:', models.status())
models.load_encoder_models()
models.load_decoder_models()
models._ensure_lcm_lora()
print('\n✅ 全モデル GPU 常駐完了！ Status:', models.status())

## ステップ 3：Web UI (`app.py`) を起動してブラウザで開く
下のセルを実行すると、Cloudflare Tunnel 経由で **公開 URL（`https://xxxx.trycloudflare.com`）** が発行されます。
表示されたリンクをクリックすると、手元のPCのブラウザで **SemanticCodec v1.0.0 Web UI** が開き、A100 GPU でリアルタイムにエンコード・デコード（Turbo / Standard / HighFi）を操作できます！

In [ ]:
import subprocess, time, re, threading
from google.colab import output

# Cloudflared バイナリの取得（SSE 進捗バーと大容量ファイル送受信が最も安定します）
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Cloudflared トンネルをバックグラウンドで起動
tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break
    time.sleep(0.25)

print('=' * 70)
if public_url:
    print(f'🌐 以下の URL をクリックして SemanticCodec v1.0.0 Web UI を開いてください:')
    print(f'👉 {public_url}')
else:
    print('Colab ポート転送リンクを表示します:')
    output.serve_kernel_port_as_window(5000)
print('=' * 70)
print('※ 終了するときはこのセルの停止ボタン（■）を押し、ステップ 5 でランタイムを切断してください。\n')

# Flask サーバーの起動（プリロード済みモデルをそのまま利用）
import app as flask_app
if 'models' in globals():
    flask_app.MODELS = models
flask_app.app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False, threaded=True)

## （オプション）ステップ 4：CLI ベンチマーク・コマンドを直接試す場合
Web UI ではなく CLI（`cli.py`）で手元の画像のベンチマーク表（`Lite` / `Standard` / `HighFi`）やデコードを一括実行したい場合は、下のセルを使えます。

In [ ]:
from google.colab import files

print('テストしたい画像（JPG / PNG）を1枚選択してください:')
up = files.upload()
if up:
    img_name = list(up.keys())[0]
    !python cli.py benchmark "{img_name}"
    !python cli.py encode "{img_name}" --mode Standard -o test_out.semic.jpg
    !python cli.py decode test_out.semic.jpg --mode Turbo -o decoded_turbo.jpg
    !python cli.py decode test_out.semic.jpg --mode Standard -o decoded_standard.jpg
    print('\n生成されたファイルをダウンロードします...')
    files.download('test_out.semic.jpg')
    files.download('decoded_turbo.jpg')
    files.download('decoded_standard.jpg')

## 🛑 ステップ 5：ランタイムを完全停止して課金カウントを終了する
**テストが終わったら必ず下のセルを実行してください。**
A100 ランタイムの割り当てが即座に解除され、コンピューティングユニットの消費がピタッと止まります。

In [ ]:
from google.colab import runtime
print('A100 ランタイムを接続解除・削除しています（課金カウントを停止します）...')
runtime.unassign()